# Cheap-Talk Benchmark - Kaggle Runner

Runs the full Sabani/Georgousis-aligned sweep (5 runs x 16 rounds x {PD,SH}) for ONE model on Kaggle's free 30 h/week T4 GPU.

**Scenarios covered (each one a separate command in cells 6-12 below):**
1. Baseline: meaningful cheap-talk vs no-comm
2. No-sense (channel-only ablation)
3. Silence (empty channel)
4. Counterfactual (IF/WOULD framing)
5. Framing-business / -team / -competitive (Lore & Heydari)

**Before running:**
1. Settings (right panel) -> Accelerator -> `GPU T4 x2`
2. Settings -> Internet -> `On`
3. Add Kaggle Secret named `HF_TOKEN` if you'll use a gated model (Llama, Gemma)
4. Edit `MODEL` in cell 4 below

## Cell 1 - Install missing deps (DO NOT upgrade torch/transformers!)

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), "GPU not enabled! Settings -> Accelerator -> GPU T4 x2"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import transformers
print(f"torch: {torch.__version__}  transformers: {transformers.__version__}")

## Cell 2 - Clone the benchmark code

In [ ]:
GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"

import os
if not os.path.exists('/kaggle/working/repo'):
    !git clone $GITHUB_REPO /kaggle/working/repo
%cd /kaggle/working/repo/cheaptalk_bench
!ls

## Cell 3 - HF token (only for gated Llama / Gemma)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded from Kaggle Secrets')
except Exception as e:
    print(f'No HF_TOKEN (fine for Qwen): {e}')

## Cell 4 - Pick the model

ONE model per session. Each scenario below reuses the same loaded model.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
# MODEL = "Qwen/Qwen2.5-3B-Instruct"
# MODEL = "Qwen/Qwen2.5-14B-Instruct"
# MODEL = "google/gemma-2-9b-it"      # gated -- needs HF token
# MODEL = "meta-llama/Llama-3.1-8B-Instruct"  # gated -- needs Meta approval

MODEL_SHORT = MODEL.split('/')[-1]
BASE_DIR = f"results/{MODEL_SHORT}"
print(f"Model: {MODEL}")
print(f"Outputs root: {BASE_DIR}/<scenario>/")

## Cell 5 - Smoke test (5-10 min, 2x8 instead of 5x16)

Verifies model loads + JSON parses. Run once per model before committing GPU time.

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --out-dir $BASE_DIR/_smoke --quick --no-probe

## Cell 6 - Scenario 1: Baseline (no_comm + meaningful cheap_talk)

This is the canonical Sabani/Georgousis comparison. ~1.5-2h on T4 for 7-9B.

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --out-dir $BASE_DIR/baseline --no-probe

## Cell 7 - Scenario 2: No-sense (channel-only)

RQ1 control: do agents react to canned off-topic small-talk the same way they react to meaningful messages? Only cheap_talk is needed (no_comm is already saved in scenario 1).

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --message-policy no_sense --conditions cheap_talk \
    --out-dir $BASE_DIR/no_sense --no-probe

## Cell 8 - Scenario 3: Silence (empty channel)

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --message-policy silence --conditions cheap_talk \
    --out-dir $BASE_DIR/silence --no-probe

## Cell 9 - Scenario 4: Counterfactual messages

Agents instructed to phrase messages as IF/WOULD counterfactuals. Tests whether causal/hypothetical reasoning amplifies cooperation.

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --message-policy counterfactual --conditions cheap_talk \
    --out-dir $BASE_DIR/counterfactual --no-probe

## Cell 10 - Scenario 5a: Framing - business

Lore & Heydari (2024) framing: agents talk like business partners (investment, ROI, contract...).

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --message-policy framing --framing-type business --conditions cheap_talk \
    --out-dir $BASE_DIR/framing_business --no-probe

## Cell 11 - Scenario 5b: Framing - team

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --message-policy framing --framing-type team --conditions cheap_talk \
    --out-dir $BASE_DIR/framing_team --no-probe

## Cell 12 - Scenario 5c: Framing - competitive

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL \
    --message-policy framing --framing-type competitive --conditions cheap_talk \
    --out-dir $BASE_DIR/framing_competitive --no-probe

## Cell 13 - Pack all results into one zip

In [ ]:
import shutil
zip_path = f"/kaggle/working/results_{MODEL_SHORT}"
shutil.make_archive(zip_path, 'zip', BASE_DIR)
print(f"Done. Zip created at {zip_path}.zip")
!ls -lh /kaggle/working/*.zip

## Cell 14 - On-Kaggle quick analysis (optional)

Note: analysis.py only knows about no_comm/cheap_talk + message_ablation/. The new scenarios are in $BASE_DIR/<scenario>/cheap_talk/, so for now run analysis per scenario:

In [ ]:
for scenario in ['baseline', 'no_sense', 'silence', 'counterfactual',
                 'framing_business', 'framing_team', 'framing_competitive']:
    print(f'\n========== {scenario} ==========')
    !python analysis.py --results-dir $BASE_DIR/$scenario || echo '(no data for this scenario)'